# Final model and feature-selection comparison

This notebook is a **read-only report**. It loads completed out-of-fold artifacts from the global SRM, patient-adaptive, FusionMLP, and PairModel notebooks. It does not fit, tune, or select model parameters.

The principal comparison asks whether control-aware feature selection reduces healthy-control change while retaining strong and consistent FRDA progression.

## 1. Result provenance and completeness

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eval.recipe_models import comparison_table, frda_control_contrast_table
from src.reporting.experiment_artifacts import read_experiment_contract, read_table_artifact

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
OUTPUT_DIR = RUN_DIR / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest, folds = read_experiment_contract(RUN_DIR)

MODEL_DIRS = {
    "Global linear SRM": RUN_DIR / "models" / "srm_global_linear",
    "Patient-adaptive": RUN_DIR / "models" / "patient_adaptive",
    "Deep learning": RUN_DIR / "models" / "deep_learning",
}
performance_parts = []
oof_parts = []
completeness = []
for family, directory in MODEL_DIRS.items():
    performance_path = directory / "performance.csv"
    oof_path = directory / "oof_visit_scores.csv"
    if not performance_path.exists() or not oof_path.exists():
        raise FileNotFoundError(f"Run the {family} notebook first: {directory}")
    performance = read_table_artifact(performance_path, schema="performance", manifest=manifest)
    oof = read_table_artifact(oof_path, schema="oof_visit_scores", manifest=manifest)
    performance_parts.append(performance)
    oof_parts.append(oof)
    completeness.append({
        "Model family": family, "Performance rows": len(performance), "OOF visit rows": len(oof),
        "Strategies": ", ".join(sorted(performance["selection_strategy"].unique())),
        "Cohorts": ", ".join(sorted(performance["cohort"].unique())), "Run ID": performance["run_id"].iloc[0],
    })

all_performance = pd.concat(performance_parts, ignore_index=True)
all_oof = pd.concat(oof_parts, ignore_index=True)
if set(all_performance["run_id"]) != {RUN_ID} or set(all_oof["run_id"]) != {RUN_ID}:
    raise ValueError("Mixed or stale run IDs detected")
display(pd.DataFrame(completeness))
display(pd.DataFrame([{
    "Run ID": RUN_ID, "Data SHA256": manifest["data_sha256"][:12],
    "Feature-panel SHA256": manifest["feature_panel_sha256"][:12],
    "Outer folds": manifest["outer_splits"], "Seed": manifest["seed"],
}]))

,Model family,Performance rows,OOF visit rows,Strategies,Cohorts,Run ID
0,Global linear SRM,12,1332,"control_aware, frda_only","Control, FRDA",trackfa_70_feature_comparison_v1
1,Patient-adaptive,24,2664,"control_aware, frda_only","Control, FRDA",trackfa_70_feature_comparison_v1
2,Deep learning,24,2664,"control_aware, frda_only","Control, FRDA",trackfa_70_feature_comparison_v1


,Run ID,Data SHA256,Feature-panel SHA256,Outer folds,Seed
0,trackfa_70_feature_comparison_v1,2064ae05cec5,bdc4ab494158,5,42


## 2. Primary held-out comparison

In [2]:
contrast = frda_control_contrast_table(all_oof, n_boot=1000, seed=int(manifest["seed"]))
run_metadata = (all_oof.groupby(["model", "selection_strategy"], as_index=False)
    .agg(feature_count=("feature_count", "max"),
         modulators=("modulator_recipe", lambda values: "; ".join(sorted(set(values.astype(str)))))))
site_summary_parts = []
for directory in MODEL_DIRS.values():
    site_path = directory / "site_diagnostics.csv"
    if site_path.exists():
        site_summary_parts.append(pd.read_csv(site_path))
site_summary = pd.concat(site_summary_parts, ignore_index=True) if site_summary_parts else pd.DataFrame()
if site_summary.empty:
    site_flags = pd.DataFrame(columns=["model", "selection_strategy", "site_flag"])
else:
    site_flags = (site_summary.groupby(["model", "selection_strategy"], as_index=False)
        .agg(site_flag=("site_p_value", lambda values: "flagged" if (pd.to_numeric(values, errors="coerce") < 0.05).any() else "not flagged")))
headline = comparison_table(all_performance).merge(
    contrast[[
        "model", "selection_strategy", "contrast_ci_low", "contrast_ci_high", "p_contrast_gt_0",
        "frda_n_participants", "frda_n_pairs", "control_n_participants", "control_n_pairs",
    ]],
    on=["model", "selection_strategy"],
    how="left",
)
headline = headline.merge(run_metadata, on=["model", "selection_strategy"], how="left")
headline = headline.merge(site_flags, on=["model", "selection_strategy"], how="left")
labels = {
    "srm_global_linear": "Global linear SRM",
    "patient_adaptive_strategy_specific": "Patient-adaptive: strategy-specific modulator",
    "patient_adaptive_common_modulator": "Patient-adaptive: common modulator",
    "fusion_mlp": "FusionMLP",
    "pair_model": "PairModel",
}
headline["Model"] = headline["model"].map(labels).fillna(headline["model"])
headline["Feature selection"] = headline["selection_strategy"].map({
    "frda_only": "FRDA-only", "control_aware": "Control-aware",
})

primary_columns = [
    "Model", "Feature selection", "feature_count", "modulators", "site_flag",
    "frda_pooled_annual_d_z", "frda_pooled_annual_ci_low", "frda_pooled_annual_ci_high",
    "frda_n_participants", "frda_n_pairs",
    "control_pooled_annual_d_z", "control_pooled_annual_ci_low", "control_pooled_annual_ci_high",
    "control_n_participants", "control_n_pairs",
    "signed_frda_control_contrast", "contrast_ci_low", "contrast_ci_high", "p_contrast_gt_0",
    "absolute_control_d_z", "frda_interval_gap",
]
primary = headline[[column for column in primary_columns if column in headline]].sort_values(
    ["frda_pooled_annual_d_z", "absolute_control_d_z"], ascending=[False, True], kind="mergesort"
)
print("Primary comparison: pooled annual progression and disease specificity")
display(primary.round(3))
primary.to_csv(OUTPUT_DIR / "primary_model_comparison.csv", index=False)
contrast.to_csv(OUTPUT_DIR / "frda_control_contrasts.csv", index=False)

Primary comparison: pooled annual progression and disease specificity


,Model,Feature selection,feature_count,modulators,site_flag,frda_pooled_annual_d_z,frda_pooled_annual_ci_low,frda_pooled_annual_ci_high,frda_n_participants,frda_n_pairs,...,control_pooled_annual_ci_low,control_pooled_annual_ci_high,control_n_participants,control_n_pairs,signed_frda_control_contrast,contrast_ci_low,contrast_ci_high,p_contrast_gt_0,absolute_control_d_z,frda_interval_gap
8,Global linear SRM,Control-aware,16,none,flagged,0.749,0.614,0.900,117,207,...,-0.168,0.168,67,126,0.760,0.594,0.934,1.000,0.011,0.365
9,Global linear SRM,FRDA-only,16,none,flagged,0.749,0.610,0.912,117,207,...,-0.121,0.222,67,126,0.715,0.529,0.925,1.000,0.033,0.390
5,Patient-adaptive: common modulator,FRDA-only,16,disease_duration; gaa_1,flagged,0.709,0.570,0.882,117,207,...,-0.057,0.285,67,126,0.595,0.407,0.783,1.000,0.114,0.172
4,Patient-adaptive: common modulator,Control-aware,16,disease_duration; gaa_1,flagged,0.662,0.502,0.832,117,207,...,-0.098,0.262,67,126,0.586,0.411,0.773,1.000,0.076,0.270
6,Patient-adaptive: strategy-specific modulator,Control-aware,16,"disease_duration; disease_duration,gaa_1; gaa_1",flagged,0.662,0.502,0.830,117,207,...,-0.099,0.265,67,126,0.586,0.412,0.771,1.000,0.076,0.256
7,Patient-adaptive: strategy-specific modulator,FRDA-only,16,"disease_duration; disease_duration,gaa_1; gaa_1",flagged,0.654,0.504,0.826,117,207,...,-0.086,0.244,67,126,0.579,0.398,0.771,1.000,0.076,0.150
2,PairModel,Control-aware,16,none,not flagged,0.297,0.166,0.434,117,207,...,-0.249,0.128,67,126,0.339,0.177,0.513,1.000,0.042,0.196
0,FusionMLP,Control-aware,16,none,flagged,0.263,0.136,0.385,117,207,...,-0.079,0.272,67,126,0.175,-0.003,0.364,0.972,0.089,0.237
1,FusionMLP,FRDA-only,16,none,not flagged,0.247,0.119,0.366,117,207,...,-0.107,0.242,67,126,0.183,0.026,0.362,0.989,0.064,0.211
3,PairModel,FRDA-only,16,none,not flagged,0.174,0.038,0.310,117,207,...,-0.051,0.259,67,126,0.058,-0.135,0.258,0.727,0.116,0.153


## 3. Interval-specific comparison

In [3]:
interval_columns = [
    "Model", "Feature selection",
    "frda_v1_v2_d_z", "frda_v1_v2_ci_low", "frda_v1_v2_ci_high", "frda_v1_v2_n_pairs",
    "frda_v2_v3_d_z", "frda_v2_v3_ci_low", "frda_v2_v3_ci_high", "frda_v2_v3_n_pairs",
    "control_v1_v2_d_z", "control_v1_v2_ci_low", "control_v1_v2_ci_high", "control_v1_v2_n_pairs",
    "control_v2_v3_d_z", "control_v2_v3_ci_low", "control_v2_v3_ci_high", "control_v2_v3_n_pairs",
    "frda_interval_gap",
]
interval_table = headline[[column for column in interval_columns if column in headline]].sort_values(
    ["Model", "Feature selection"], kind="mergesort"
)
display(interval_table.round(3))
interval_table.to_csv(OUTPUT_DIR / "interval_model_comparison.csv", index=False)

,Model,Feature selection,frda_v1_v2_d_z,frda_v1_v2_ci_low,frda_v1_v2_ci_high,frda_v1_v2_n_pairs,frda_v2_v3_d_z,frda_v2_v3_ci_low,frda_v2_v3_ci_high,frda_v2_v3_n_pairs,control_v1_v2_d_z,control_v1_v2_ci_low,control_v1_v2_ci_high,control_v1_v2_n_pairs,control_v2_v3_d_z,control_v2_v3_ci_low,control_v2_v3_ci_high,control_v2_v3_n_pairs,frda_interval_gap
0,FusionMLP,Control-aware,0.379,0.240,0.508,108,0.142,-0.043,0.350,99,0.140,-0.110,0.336,63,0.023,-0.235,0.253,63,0.237
1,FusionMLP,FRDA-only,0.341,0.210,0.455,108,0.130,-0.049,0.384,99,0.089,-0.155,0.331,63,0.037,-0.209,0.275,63,0.211
8,Global linear SRM,Control-aware,0.944,0.740,1.205,108,0.579,0.387,0.812,99,-0.019,-0.257,0.248,63,-0.003,-0.268,0.269,63,0.365
9,Global linear SRM,FRDA-only,0.959,0.754,1.234,108,0.569,0.383,0.805,99,0.007,-0.242,0.265,63,0.059,-0.189,0.343,63,0.390
2,PairModel,Control-aware,0.196,0.007,0.384,108,0.392,0.235,0.545,99,-0.244,-0.492,-0.009,63,0.121,-0.106,0.345,63,0.196
3,PairModel,FRDA-only,0.251,0.070,0.441,108,0.098,-0.086,0.295,99,0.054,-0.185,0.317,63,0.159,-0.111,0.362,63,0.153
4,Patient-adaptive: common modulator,Control-aware,0.802,0.556,1.104,108,0.532,0.348,0.752,99,0.124,-0.114,0.391,63,0.032,-0.227,0.278,63,0.270
5,Patient-adaptive: common modulator,FRDA-only,0.798,0.594,1.053,108,0.625,0.439,0.837,99,0.163,-0.070,0.436,63,0.068,-0.206,0.337,63,0.172
6,Patient-adaptive: strategy-specific modulator,Control-aware,0.793,0.552,1.069,108,0.537,0.356,0.753,99,0.136,-0.105,0.400,63,0.021,-0.247,0.267,63,0.256
7,Patient-adaptive: strategy-specific modulator,FRDA-only,0.728,0.527,0.988,108,0.577,0.408,0.769,99,0.085,-0.160,0.345,63,0.066,-0.199,0.339,63,0.150


## 4. Effect of control-aware feature selection

In [4]:
difference_fields = [
    "frda_pooled_annual_d_z", "control_pooled_annual_d_z", "signed_frda_control_contrast",
    "absolute_control_d_z", "frda_v1_v2_d_z", "frda_v2_v3_d_z", "frda_interval_gap",
]
wide = headline.pivot_table(index=["model", "Model"], columns="selection_strategy", values=difference_fields, aggfunc="first")
differences = pd.DataFrame(index=wide.index)
for field in difference_fields:
    if (field, "control_aware") in wide and (field, "frda_only") in wide:
        differences[f"change_in_{field}"] = wide[(field, "control_aware")] - wide[(field, "frda_only")]
differences = differences.reset_index().drop(columns="model")
print("Control-aware minus FRDA-only; positive FRDA/contrast and negative absolute-control changes are favourable")
display(differences.round(3))
differences.to_csv(OUTPUT_DIR / "feature_strategy_differences.csv", index=False)

Control-aware minus FRDA-only; positive FRDA/contrast and negative absolute-control changes are favourable


,Model,change_in_frda_pooled_annual_d_z,change_in_control_pooled_annual_d_z,change_in_signed_frda_control_contrast,change_in_absolute_control_d_z,change_in_frda_v1_v2_d_z,change_in_frda_v2_v3_d_z,change_in_frda_interval_gap
0,FusionMLP,0.016,0.025,-0.008,0.025,0.038,0.012,0.026
1,PairModel,0.123,-0.158,0.281,-0.073,-0.055,0.294,0.043
2,Patient-adaptive: common modulator,-0.047,-0.038,-0.009,-0.038,0.005,-0.093,0.098
3,Patient-adaptive: strategy-specific modulator,0.007,0.000,0.007,0.000,0.066,-0.040,0.106
4,Global linear SRM,0.001,-0.044,0.045,-0.022,-0.015,0.009,-0.025


## 5. Selected features and adaptive modulators

In [5]:
selection_comparison = pd.read_csv(RUN_DIR / "selections" / "feature_recipe_comparison.csv")
print("Feature-selection overlap by outer fold")
display(selection_comparison)

modulator_frequency = pd.read_csv(RUN_DIR / "models" / "patient_adaptive" / "modulator_frequency.csv")
modulator_frequency["Feature selection"] = modulator_frequency["selection_strategy"].map({
    "frda_only": "FRDA-only", "control_aware": "Control-aware",
})
print("Patient-adaptive modulator selection frequency from inner FRDA validation")
display(modulator_frequency[[
    "comparison_mode", "Feature selection", "modulators", "folds_selected", "selection_frequency",
]].sort_values(["comparison_mode", "Feature selection", "selection_frequency"], ascending=[True, True, False]).round(3))

Feature-selection overlap by outer fold


,outer_fold,baseline_features,control_aware_features,shared_features,jaccard,removed_from_baseline,added_by_control_aware
0,1,16,16,14,0.777778,Caudate | RD_SCP,FA_bCC | RD_sCC
1,2,16,16,12,0.600000,Caudate | Medulla | Putamen | sCSA_C12_UMN,FA_PCR | Pallidum | RD_PTR | RD_bCC
2,3,16,16,14,0.777778,RD_sCC | SCP,FA_ILF_IFOF | Pallidum
3,4,16,16,13,0.684211,SCP | TotalBrainWMVol_nocereb | sCSA_C12_UMN,FA_Fx_ST | FA_RLIC | FA_bCC
4,5,16,16,14,0.777778,Caudate | FA_RLIC,FA_ILF_IFOF | sCSA_C12_UMN


Patient-adaptive modulator selection frequency from inner FRDA validation


,comparison_mode,Feature selection,modulators,folds_selected,selection_frequency
1,common_modulator,Control-aware,gaa_1,3,0.6
0,common_modulator,Control-aware,disease_duration,2,0.4
3,common_modulator,FRDA-only,gaa_1,3,0.6
2,common_modulator,FRDA-only,disease_duration,2,0.4
4,strategy_specific,Control-aware,disease_duration,2,0.4
6,strategy_specific,Control-aware,gaa_1,2,0.4
5,strategy_specific,Control-aware,"disease_duration,gaa_1",1,0.2
8,strategy_specific,FRDA-only,"disease_duration,gaa_1",2,0.4
9,strategy_specific,FRDA-only,gaa_1,2,0.4
7,strategy_specific,FRDA-only,disease_duration,1,0.2


## 6. Site diagnostics and interpretation

In [6]:
site_parts = []
for family, directory in MODEL_DIRS.items():
    path = directory / "site_diagnostics.csv"
    if path.exists():
        part = pd.read_csv(path)
        part.insert(0, "model_family", family)
        site_parts.append(part)
site_diagnostics = pd.concat(site_parts, ignore_index=True) if site_parts else pd.DataFrame()
if site_diagnostics.empty:
    print("No estimable site diagnostics were available.")
else:
    columns = [column for column in [
        "model_family", "model", "selection_strategy", "cohort", "n", "site_levels",
        "site_r2_delta", "site_p_value",
    ] if column in site_diagnostics]
    display(site_diagnostics[columns].round(3))
    site_diagnostics.to_csv(OUTPUT_DIR / "site_diagnostics.csv", index=False)

print("Comparator artifacts:", OUTPUT_DIR)

,model_family,model,selection_strategy,cohort,n,site_levels,site_r2_delta,site_p_value
0,Global linear SRM,srm_global_linear,frda_only,FRDA,207,6,0.075,0.007
1,Global linear SRM,srm_global_linear,frda_only,Control,126,6,0.052,0.258
2,Global linear SRM,srm_global_linear,control_aware,FRDA,207,6,0.077,0.006
3,Global linear SRM,srm_global_linear,control_aware,Control,126,6,0.047,0.315
4,Patient-adaptive,patient_adaptive_strategy_specific,frda_only,FRDA,207,6,0.070,0.012
5,Patient-adaptive,patient_adaptive_strategy_specific,frda_only,Control,126,6,0.030,0.586
6,Patient-adaptive,patient_adaptive_strategy_specific,control_aware,FRDA,207,6,0.066,0.017
7,Patient-adaptive,patient_adaptive_strategy_specific,control_aware,Control,126,6,0.020,0.791
8,Patient-adaptive,patient_adaptive_common_modulator,frda_only,FRDA,207,6,0.068,0.014
9,Patient-adaptive,patient_adaptive_common_modulator,frda_only,Control,126,6,0.033,0.542


Comparator artifacts: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/comparison


## Reading the comparison

- **FRDA sensitivity:** larger positive pooled and interval-specific (d_z) is better.
- **Control specificity:** an effect close to zero, or opposite to the FRDA direction, is desirable.
- **Signed contrast:** positive values mean stronger progression in FRDA than controls; its bootstrap interval directly quantifies the group separation.
- **Interval gap:** smaller values indicate more consistent annual sensitivity across V1→V2 and V2→V3.
- **Model complexity:** the simpler global SRM should remain preferred unless a more complex adaptive or neural model provides a clear, reproducible improvement.

The comparator reports results; it does not use these outer-test values to retune models.